In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello World: วงจรควอนตัมแรกของคุณ

สร้าง [Bell state](https://en.wikipedia.org/wiki/Bell_state) (qubit ที่พัวพันกันสองตัว) แล้วรันด้วยสามวิธี:

1. **การจำลองแบบ Ideal** — ผลลัพธ์สมบูรณ์แบบ ไม่ต้องมีบัญชี
2. **การจำลองแบบ Noisy** — จำลองอุปกรณ์จริง ไม่ต้องมีบัญชี
3. **ฮาร์ดแวร์ควอนตัมจริง** — ต้องมี [บัญชี IBM Quantum](https://janlahmann.github.io/Qiskit-documentation/#setting-up-your-ibm-quantum-account)

## สร้าง Circuit

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## ตัวเลือกที่ 1: การจำลองแบบ Ideal (ไม่ต้องมีบัญชี)
ใช้ `StatevectorSampler` — simulator ในเครื่องที่ให้ผลลัพธ์สมบูรณ์แบบ ปราศจาก noise

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## ตัวเลือกที่ 2: การจำลองแบบ Noisy (ไม่ต้องมีบัญชี)
ใช้ `FakeManilaV2` — simulator ในเครื่องที่เลียนแบบอุปกรณ์ควอนตัม IBM จริง รวมถึงคุณสมบัติ noise ด้วย วงจรจะต้องถูก transpile (ปรับให้เข้ากัน) กับชุด Gate และการเชื่อมต่อ qubit ของอุปกรณ์ก่อน

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## ตัวเลือกที่ 3: ฮาร์ดแวร์ควอนตัมจริง
ต้องมีบัญชี IBM Quantum ดูรายละเอียดได้ที่ [การตั้งค่าบัญชี IBM Quantum](https://janlahmann.github.io/Qiskit-documentation/#setting-up-your-ibm-quantum-account)

ถ้ายังไม่ได้บันทึก credentials ในเซสชัน Binder นี้ ให้รัน cell นี้ก่อน:

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="<your-api-key>",
    instance="<your-crn>",
    overwrite=True
)

**หมายเหตุ:** งานบนฮาร์ดแวร์จริงอาจใช้เวลาสักพักขึ้นอยู่กับคิวรอ ถ้า cell ยังรันอยู่ สามารถตรวจสอบสถานะงานและดูผลลัพธ์ได้ที่ [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me)

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## ขั้นตอนถัดไป
- **[Tutorials](https://mybinder.org/v2/gh/JanLahmann/Qiskit-documentation/main?filepath=docs/tutorials)** — คู่มือทีละขั้นตอนเกี่ยวกับ algorithm, การลด error, transpilation และอื่น ๆ
- **[Courses](https://mybinder.org/v2/gh/JanLahmann/Qiskit-documentation/main?filepath=learning/courses)** — เส้นทางการเรียนรู้แบบมีโครงสร้าง ตั้งแต่พื้นฐานควอนตัมไปจนถึงการคำนวณระดับ utility
- **[Local testing mode](https://janlahmann.github.io/Qiskit-documentation/#no-token-use-local-testing-mode)** — รัน notebook ส่วนใหญ่ได้โดยไม่ต้องมีบัญชี IBM Quantum